# माइक्रोसॉफ्ट एजेंट फ्रेमवर्क — Azure OpenAI (Responses API)

इस कोड नमूने में, आप **माइक्रोसॉफ्ट एजेंट फ्रेमवर्क (MAF)** का उपयोग करके एक सरल एजेंट बनाएंगे जो **Azure OpenAI** द्वारा समर्थित होगा, और इसके लिए **Responses API** का उपयोग करेंगे।

> **माइग्रेशन नोट:** इस नमूने में पहले Semantic Kernel के साथ GitHub Models का उपयोग किया गया था। इसे माइक्रोसॉफ्ट एजेंट फ्रेमवर्क में माइग्रेट किया गया है, और GitHub Models (जो अवकाश ग्रहण कर रहा है, जुलाई 2026 में सेवानिवृत्त होगा) को Azure OpenAI से बदला गया है, जो Responses API का समर्थन करता है। MAF में `OpenAIChatClient` Azure OpenAI के स्थिर `/openai/v1/` एंडपॉइंट को लक्षित करता है और डिफ़ॉल्ट रूप से Responses API का उपयोग करता है।

इस नमूने का उद्देश्य उन चरणों को प्रदर्शित करना है जिन्हें बाद में अन्य कोड नमूनों में विभिन्न एजेंटिक पैटर्न लागू करते समय उपयोग किया जाएगा।


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## आवश्यक पायथन पैकेज आयात करें


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## एक टूल को परिभाषित करना

Microsoft Agent Framework में, एक **टूल** एक साधारण Python फ़ंक्शन होता है जिसे `@tool` से सजाया गया होता है जिसे एजेंट कॉल कर सकता है। नीचे हम एक टूल को परिभाषित करते हैं जो एक यादृच्छिक छुट्टी गंतव्य लौटाता है और पिछले गंतव्य को दोहराने से बचता है।


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## एजेंट बनाना

यहाँ, हम `TravelAgent` नामक एजेंट बनाते हैं।

इस उदाहरण में, हम बहुत ही बुनियादी निर्देशों का उपयोग करते हैं। एजेंट के व्यवहार में बदलाव देखने के लिए इन निर्देशों को संशोधित करने के लिए स्वतंत्र महसूस करें।


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## एजेंट चलाना

अब हम एजेंट को चला सकते हैं। हम एक `AgentSession` बनाते हैं ताकि एजेंट बातचीत को मोड़ों के बीच याद रख सके, फिर दो `user_inputs` भेजते हैं। पहला एक यात्रा के लिए पूछता है; दूसरा कहता है कि उपयोगकर्ता को सुझाव पसंद नहीं आया और एक और मांगता है — एजेंट सेशन इतिहास और `get_random_destination` टूल का उपयोग कर प्रतिक्रिया देता है।

आप इन संदेशों को संशोधित कर सकते हैं यह देखने के लिए कि एजेंट कैसे अलग प्रतिक्रिया देता है। प्रतिक्रियाएँ **टोकन-दर-टोकन** स्ट्रीम की जाती हैं।


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
